In [ ]:
# -*- coding: utf-8 -*-
"""
Universality Verification on CWRU Dataset
Model: 1D-Transformer 
Evaluation: 5 Independent Runs (Mean ± Std) for Delta T Analysis
Task: Evaluate Accuracy vs Delta T (1 to 10 Rotation Periods)
"""

import os
import random
import warnings
import numpy as np
import scipy.io as sio

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')

# ==========================================================================
# 0. GLOBAL TYPOGRAPHY 
# ==========================================================================
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 12,
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

# ==========================================================================
# 1. CONFIGURATION
# ==========================================================================
_FS = 12000 
# Calculate delta T for 1 to 10 rotation periods (1797 RPM -> ~0.0334s per period)
_DELTA_T_SEC_LIST = [round(0.0334 * i, 4) for i in range(1, 11)]

class CFG:
    DATA_ROOT   = r'F:\CWRU\matfiles'
    FILE_NAMES  = [
        '0_0.mat', 
        '7_1.mat', '7_2.mat', '7_3.mat',
        '14_1.mat', '14_2.mat', '14_3.mat',
        '21_1.mat', '21_2.mat', '21_3.mat'
    ]
    NUM_CLASSES = 10
    
    TOTAL_POINTS  = 119808
    SAMPLE_LENGTH = 2048
    SPLIT_RATIO   = (0.7, 0.2, 0.1)
    
    FS = _FS 
    OVERLAP_RATES = [0.0, 0.25, 0.5, 0.75, 0.9]
    
    DELTA_T_SEC_LIST = _DELTA_T_SEC_LIST
    # Convert seconds to data points (Delta T * Fs)
    DELTA_T_LIST = [int(t * _FS) for t in _DELTA_T_SEC_LIST]
    
    SEEDS         = [42, 123, 456, 789, 1010]
    
    BATCH_SIZE    = 64
    EPOCHS        = 100
    LR            = 1e-3
    WEIGHT_DECAY  = 1e-4
    DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ==========================================================================
# 2. DATA LOADER
# ==========================================================================
def _load_cwru_mat(path):
    mat = sio.loadmat(path)
    for key, value in mat.items():
        if not key.startswith('__'):
            if 'DE_time' in key or (isinstance(value, np.ndarray) and value.size > 10000):
                return np.asarray(value).astype(np.float32).ravel()
    raise ValueError(f"No valid signal found in {path}")

def load_raw_signals():
    print(f"Loading CWRU dataset from: {CFG.DATA_ROOT}")
    raw = {}
    for label, fname in enumerate(CFG.FILE_NAMES):
        path = os.path.join(CFG.DATA_ROOT, fname)
        sig = _load_cwru_mat(path)
        if sig.shape[0] < CFG.TOTAL_POINTS:
            raise ValueError(f'{fname} length {sig.shape[0]} < {CFG.TOTAL_POINTS}')
        raw[label] = sig[:CFG.TOTAL_POINTS]
    print("Successfully loaded all 10 classes.\n")
    return raw

# ==========================================================================
# 3. SLICING & DATASET
# ==========================================================================
def slice_signal(signal, win, overlap):
    step = max(1, int(round(win * (1 - overlap))))
    starts = np.arange(0, len(signal) - win + 1, step)
    samples = np.stack([signal[s:s+win] for s in starts], axis=0)
    return samples, starts

def zscore(x):
    mu, sd = x.mean(axis=1, keepdims=True), x.std(axis=1, keepdims=True) + 1e-8
    return (x - mu) / sd

def build_dataset(raw, overlap):
    X, y, starts = [], [], []
    for label, sig in raw.items():
        samp, st = slice_signal(sig, CFG.SAMPLE_LENGTH, overlap)
        X.append(samp)
        y.append(np.full(len(samp), label))
        starts.append(st)
    return zscore(np.concatenate(X)), np.concatenate(y), np.concatenate(starts)

def split_indices(X, y, starts, strategy, seed, ratios, gap):
    rng = np.random.RandomState(seed)
    N = len(X)
    train_r, val_r, test_r = ratios

    if strategy == 'random':
        idx = np.arange(N)
        rng.shuffle(idx)
        n_tr = int(N * train_r)
        n_va = int(N * val_r)
        return idx[:n_tr], idx[n_tr:n_tr+n_va], idx[n_tr+n_va:]

    if strategy in ('sequential', 'block_holdout'):
        idx_tr, idx_va, idx_te = [], [], []
        for c in np.unique(y):
            cls_idx = np.where(y == c)[0]
            cls_idx = cls_idx[np.argsort(starts[cls_idx])]
            n = len(cls_idx)
            n_tr = int(n * train_r)
            n_va = int(n * val_r)
            tr = cls_idx[:n_tr]
            va = cls_idx[n_tr:n_tr+n_va]
            te = cls_idx[n_tr+n_va:]

            if strategy == 'block_holdout' and len(tr) and len(te):
                tr_end_t = starts[tr].max() + CFG.SAMPLE_LENGTH
                va = va[starts[va] >= tr_end_t + gap] if len(va) else va
                va_end_t = starts[va].max() + CFG.SAMPLE_LENGTH if len(va) else tr_end_t
                te = te[starts[te] >= va_end_t + gap]

            idx_tr.append(tr)
            idx_va.append(va)
            idx_te.append(te)
        return (np.concatenate(idx_tr), np.concatenate(idx_va), np.concatenate(idx_te))
    raise ValueError(f"Unknown strategy: {strategy}")

class SignalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

# ==========================================================================
# 4. 1D-Transformer MODEL
# ==========================================================================
class Transformer1D(nn.Module):
    def __init__(self, num_classes=10, seq_len=2048, patch_size=16, d_model=64, nhead=8, num_layers=3, dim_feedforward=256, dropout=0.1):
        super(Transformer1D, self).__init__()
        self.patch_size = patch_size
        self.d_model = d_model
        
        # Patch Embedding: Split sequence into patches using 1D convolution
        self.patch_embed = nn.Conv1d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.num_patches = seq_len // patch_size
        
        # Learnable positional encoding
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, d_model))
        self.pos_drop = nn.Dropout(p=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)      
        x = x.transpose(1, 2)        
        
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        x = self.transformer_encoder(x) 
        x = x.mean(dim=1)            
        x = self.head(x)             
        return x

# ==========================================================================
# 5. EVALUATION & TRAINING
# ==========================================================================
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(CFG.DEVICE)
            out = model(x)
            ps.append(out.argmax(1).cpu().numpy())
            ys.append(y.numpy())
    if len(ys) == 0: return {'acc': 0.0}
    y = np.concatenate(ys)
    p = np.concatenate(ps)
    return {'acc': accuracy_score(y, p)}

def train_one_run(X, y, starts, strategy, overlap, seed, gap):
    set_seed(seed)
    idx_tr, idx_va, idx_te = split_indices(X, y, starts, strategy, seed, CFG.SPLIT_RATIO, gap)
    
    if seed == CFG.SEEDS[0]:
        print(f"[Test Size: {len(idx_te):>4} samples]", end=" ")
        
    if len(idx_te) == 0:
        return 0.0 

    tr_loader = DataLoader(SignalDataset(X[idx_tr], y[idx_tr]), batch_size=CFG.BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(SignalDataset(X[idx_va], y[idx_va]), batch_size=CFG.BATCH_SIZE, shuffle=False)
    te_loader = DataLoader(SignalDataset(X[idx_te], y[idx_te]), batch_size=CFG.BATCH_SIZE, shuffle=False)

    model = Transformer1D(num_classes=CFG.NUM_CLASSES, seq_len=CFG.SAMPLE_LENGTH).to(CFG.DEVICE)
    opt   = optim.Adam(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    crit  = nn.CrossEntropyLoss()

    best_va, best_state = -1, None

    for ep in range(1, CFG.EPOCHS+1):
        model.train()
        for x, yt in tr_loader:
            x, yt = x.to(CFG.DEVICE), yt.to(CFG.DEVICE)
            opt.zero_grad()
            loss = crit(model(x), yt)
            loss.backward()
            opt.step()
        
        va = evaluate(model, va_loader)
        if va['acc'] > best_va:
            best_va = va['acc']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    te_m = evaluate(model, te_loader)
    
    return te_m['acc']

def run_multi_seed_case(X, y, starts, strategy, overlap, gap):
    print(f"  Running Delta T = {gap:<4} pts ...", end=" ")
    te_accs = []
    for seed in CFG.SEEDS:
        te = train_one_run(X, y, starts, strategy, overlap, seed, gap)
        te_accs.append(te)
    
    te_m = np.mean(te_accs)
    te_std = np.std(te_accs)
    print(f"... Mean: {te_m:.4f} | Std: {te_std:.4f}")
    return te_m, te_std

# ==========================================================================
# 6. PLOTTING FUNCTION
# ==========================================================================
def plot_multi_overlap_delta_t(delta_t_sec_list, all_results_mean):
    fig, ax = plt.subplots(figsize=(9, 6))
    
    colors = ['#D64040', '#3B7DD8', '#F2A93B', '#4CAF50', '#8E44AD']
    markers = ['o', 's', '^', 'D', 'v']
    
    for i, (ov, acc_list) in enumerate(all_results_mean.items()):
        ax.plot(delta_t_sec_list, acc_list, 
                color=colors[i], linewidth=2.0, 
                marker=markers[i], markersize=7, markerfacecolor='white', 
                markeredgecolor=colors[i], markeredgewidth=1.8,
                label=f'Overlap rate = {ov}')
    
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_xlabel(r'Temporal Safety Gap $\Delta T$ (s)', fontsize=13)
    ax.set_ylabel('Test Accuracy', fontsize=13)
    
    ax.set_xticks(delta_t_sec_list)
    ax.set_xticklabels([f"{t:.4f}" for t in delta_t_sec_list], rotation=45)
    
    ax.legend(loc='lower left', fontsize=11, frameon=True, edgecolor='black', fancybox=False)
    
    plt.tight_layout()
    save_path = 'Delta_T_Multi_Overlap_CWRU_Transformer1D.png'
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    print(f"\nPlot saved to {save_path}")

# ==========================================================================
# 7. MAIN EXECUTION
# ==========================================================================
if __name__ == '__main__':
    print("Starting Multi-Overlap Delta T Analysis")
    print("Model: 1D-Transformer | Dataset: CWRU")
    print("-" * 60)
    
    raw_data = load_raw_signals()
    
    all_results_mean = {}
    all_results_std = {}
    
    for ov in CFG.OVERLAP_RATES:
        print(f"\n--- Overlap Rate: {ov} ---")
        
        X, y, starts = build_dataset(raw_data, overlap=ov)
        print(f"Total Samples Generated: {len(X)}")
        
        acc_means = []
        acc_stds = []
        
        for gap in CFG.DELTA_T_LIST:
            mean_acc, std_acc = run_multi_seed_case(X, y, starts, strategy='block_holdout', overlap=ov, gap=gap)
            acc_means.append(mean_acc)
            acc_stds.append(std_acc)
            
        all_results_mean[ov] = acc_means
        all_results_std[ov] = acc_stds

    plot_multi_overlap_delta_t(CFG.DELTA_T_SEC_LIST, all_results_mean)
    
    # Print summary table
    print("\n" + "="*130)
    header = f"{'Model/Overlap':<15} | " + " | ".join([f"{t}s".center(8) for t in CFG.DELTA_T_SEC_LIST])
    print(header)
    print("-" * 130)
    for ov in CFG.OVERLAP_RATES:
        row_str = f"Trans (ov={ov:<3}) | "
        for m, s in zip(all_results_mean[ov], all_results_std[ov]):
            cell = f"{m:.2f}±{s:.2f}"
            row_str += cell.center(8) + " | "
        print(row_str)
    print("="*130)
